# Numerical Drift Detection Experiment

This notebook tests statistical drift detection methods for **numerical features**
in the telecom churn dataset.

Three numerical drift detection methods are implemented:

1. Kolmogorov–Smirnov Test (KS Test)
Measures whether two numerical distributions are statistically different.

Hypothesis:

H0: Train and test distributions are the same  
H1: The distributions are different

Interpretation:

p-value > 0.05 → No significant drift  
p-value ≤ 0.05 → Drift detected


2. Population Stability Index (PSI)

Measures the magnitude of distribution change between training and test datasets.

Interpretation:

PSI < 0.1 → No drift  
0.1 ≤ PSI < 0.25 → Moderate drift  
PSI ≥ 0.25 → Significant drift


3. Wasserstein Distance

Measures the distance between two probability distributions.
Also known as the **Earth Mover’s Distance**.

Higher values indicate larger distribution shifts.


This experiment:

- Loads the dataset
- Simulates a production dataset using a train/test split
- Computes KS Test, PSI, and Wasserstein Distance for numerical columns
- Generates a drift summary table

In [2]:
## imports 
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML utilities
from sklearn.model_selection import train_test_split

# Statistical tests
from scipy.stats import ks_2samp
from scipy.stats import wasserstein_distance

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## Load Dataset

In [4]:
train_df = pd.read_csv("../public_data/train.csv")
train_df.head()

,CustomerID,UserGender,UserAge,YoungAdultFlag,RetireeStatus,Married,Dependents,NumberofDependents,Country,State,LocationCity,AreaCode,Latitude,Longitude,Population,ReferredaFriend,NumberofReferrals,TenureinMonths,Offer,VoiceService,AvgMonthlyLongDistanceCharges,AdditionalLines,ConnectivityType,InternetType,DataUsageAvg,CyberSecuritySvc,CloudStorageSvc,HardwareInsurance,PrioritySupport,VideoSvc_A,VideoSvc_B,AudioSvc,UnlimitedData,Contract,DigitalInvoicing,TransactionMode,MonthlyCharge,TotalCharges,TotalRefunds,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,CustomerLifetimeValue,ChurnStatus,Month
0,1610a102a7854c5d,Male,71,No,Yes,Yes,Yes,1,United States,California,Pine Valley,91962,32.800671,-116.483363,1604,Yes,9,38,NaN,Yes,28.30,Yes,Yes,Fiber Optic,27,No,Yes,No,No,Yes,Yes,No,Yes,One Year,No,Bank Withdrawal,101.15,3741.85,0.0,0,1075.40,4817.25,4036,No,25-Jan
1,7b4b47a335af4741,Male,77,No,Yes,Yes,No,0,United States,California,Cabazon,92230,33.929812,-116.760580,2355,Yes,5,27,NaN,Yes,33.57,Yes,Yes,DSL,21,No,Yes,Yes,Yes,Yes,Yes,No,Yes,One Year,Yes,Bank Withdrawal,84.80,2309.55,0.0,0,906.39,3215.94,5352,No,25-Jan
2,81465cd8c020404b,Male,78,No,Yes,Yes,No,0,United States,California,Desert Center,92239,33.889605,-115.257009,964,Yes,7,33,NaN,Yes,25.15,Yes,Yes,Fiber Optic,20,Yes,No,Yes,Yes,Yes,Yes,No,Yes,One Year,Yes,Bank Withdrawal,109.90,3694.70,0.0,0,829.95,4524.65,5655,No,25-Jan
3,8fb085ca69574c4c,Male,74,No,Yes,No,No,0,United States,California,Desert Hot Springs,92241,33.832799,-116.250973,5529,No,0,33,Offer C,No,0.00,No,Yes,Cable,11,No,No,Yes,Yes,Yes,Yes,No,Yes,One Year,Yes,Bank Withdrawal,54.60,1803.70,0.0,0,0.00,1803.70,5322,No,25-Jan
4,d2ca70e00e1d419f,Male,71,No,Yes,Yes,Yes,1,United States,California,Morongo Valley,92256,34.097863,-116.594561,3499,Yes,9,32,Offer C,Yes,31.86,No,Yes,Fiber Optic,23,No,Yes,No,No,Yes,Yes,Yes,Yes,One Year,Yes,Mailed Check,93.95,2861.45,0.0,0,1019.52,3880.97,2844,No,25-Jan


## Define Numerical Columns

These are the numerical features in the telecom dataset.
They will be tested for distribution drift between datasets.

In [6]:
numerical_cols = [
    "UserAge",
    "NumberofDependents",
    "AreaCode",
    "Latitude",
    "Longitude",
    "Population",
    "NumberofReferrals",
    "TenureinMonths",
    "AvgMonthlyLongDistanceCharges",
    "DataUsageAvg",
    "MonthlyCharge",
    "TotalCharges",
    "TotalRefunds",
    "TotalExtraDataCharges",
    "TotalLongDistanceCharges",
    "TotalRevenue",
    "CustomerLifetimeValue"
]

## Simulate Production Dataset

Since we only have the training dataset available, we simulate a production dataset
by splitting the data.

70% → training baseline  
30% → simulated production / validation data

In [8]:
train_part, test_part = train_test_split(
    train_df,
    test_size=0.3,
    random_state=42
)

## Population Stability Index (PSI)

PSI measures how much a feature distribution has shifted between datasets.

Formula:

PSI = Σ (train_pct − test_pct) × ln(train_pct / test_pct)

For numerical features, values are first grouped into bins before calculating PSI.

In [10]:
def calculate_psi(train_series, test_series, bins=10):

    train_series = train_series.dropna()
    test_series = test_series.dropna()

    breakpoints = np.linspace(0, 100, bins + 1)
    breakpoints = np.percentile(train_series, breakpoints)

    train_bins = np.histogram(train_series, bins=breakpoints)[0] / len(train_series)
    test_bins = np.histogram(test_series, bins=breakpoints)[0] / len(test_series)

    psi = np.sum(
        (train_bins - test_bins) *
        np.log((train_bins + 1e-6) / (test_bins + 1e-6))
    )

    return psi

## Kolmogorov–Smirnov Test

The KS Test compares the cumulative distributions of two datasets.

It determines whether the distributions are statistically different.

In [12]:
def ks_test(train_series, test_series):

    statistic, p_value = ks_2samp(
        train_series.dropna(),
        test_series.dropna()
    )

    return statistic, p_value

## Drift Severity Classification

This helper function converts PSI values into drift severity categories.

In [14]:
def classify_psi(psi):

    if psi < 0.1:
        return "No Drift"

    elif psi < 0.25:
        return "Moderate Drift"

    else:
        return "Significant Drift"

## Run Drift Detection

For each numerical feature:

- Compute PSI
- Compute KS Test
- Compute Wasserstein Distance
- Evaluate drift severity
- Store results in a summary table

In [16]:
results = []

for col in numerical_cols:

    psi = calculate_psi(train_part[col], test_part[col])

    ks_stat, p_value = ks_test(train_part[col], test_part[col])

    wasserstein = wasserstein_distance(
        train_part[col].dropna(),
        test_part[col].dropna()
    )

    severity = classify_psi(psi)

    drift_flag = (psi > 0.25) or (p_value < 0.05)

    results.append({
        "Feature": col,
        "PSI": round(psi, 4),
        "PSI_Severity": severity,
        "KS_Statistic": round(ks_stat, 4),
        "p_value": round(p_value, 4),
        "Wasserstein_Distance": round(wasserstein, 4),
        "Drift_Detected": drift_flag
    })

## Drift Summary Table

In [18]:
drift_results = pd.DataFrame(results)

drift_results.sort_values("PSI", ascending=False)

,Feature,PSI,PSI_Severity,KS_Statistic,p_value,Wasserstein_Distance,Drift_Detected
15,TotalRevenue,0.0011,No Drift,0.0094,0.1425,29.5660,False
4,Longitude,0.0009,No Drift,0.0103,0.0838,0.0367,False
11,TotalCharges,0.0008,No Drift,0.0099,0.1126,25.2895,False
14,TotalLongDistanceCharges,0.0008,No Drift,0.0044,0.9407,7.6391,False
0,UserAge,0.0007,No Drift,0.0072,0.4335,0.1306,False
5,Population,0.0007,No Drift,0.0057,0.7190,181.9025,False
2,AreaCode,0.0006,No Drift,0.0109,0.0610,27.5768,False
7,TenureinMonths,0.0006,No Drift,0.0078,0.3202,0.2538,False
10,MonthlyCharge,0.0006,No Drift,0.0070,0.4692,0.2897,False
3,Latitude,0.0005,No Drift,0.0099,0.1069,0.0420,False


## Features with Detected Drift

This table shows features where drift was detected using either
PSI or KS statistical test.

In [20]:
drift_results[drift_results["Drift_Detected"] == True]

,Feature,PSI,PSI_Severity,KS_Statistic,p_value,Wasserstein_Distance,Drift_Detected


# Test

In [41]:
test_part_drift = test_part.copy()

test_part_drift["UserAge"] = test_part_drift["UserAge"] + 15

In [43]:
results = []

for col in numerical_cols:

    psi = calculate_psi(train_part[col], test_part_drift[col])

    ks_stat, p_value = ks_test(train_part[col], test_part_drift[col])

    wasserstein = wasserstein_distance(
        train_part[col].dropna(),
        test_part_drift[col].dropna()
    )

    severity = classify_psi(psi)

    drift_flag = (psi > 0.25) or (p_value < 0.05)

    results.append({
        "Feature": col,
        "PSI": round(psi, 4),
        "PSI_Severity": severity,
        "KS_Statistic": round(ks_stat, 4),
        "p_value": round(p_value, 4),
        "Wasserstein_Distance": round(wasserstein, 4),
        "Drift_Detected": drift_flag
    })

In [44]:
drift_results = pd.DataFrame(results)

drift_results.sort_values("PSI", ascending=False)

,Feature,PSI,PSI_Severity,KS_Statistic,p_value,Wasserstein_Distance,Drift_Detected
0,UserAge,2.5736,Significant Drift,0.3639,0.0000,14.8781,True
15,TotalRevenue,0.0011,No Drift,0.0094,0.1425,29.5660,False
4,Longitude,0.0009,No Drift,0.0103,0.0838,0.0367,False
11,TotalCharges,0.0008,No Drift,0.0099,0.1126,25.2895,False
14,TotalLongDistanceCharges,0.0008,No Drift,0.0044,0.9407,7.6391,False
5,Population,0.0007,No Drift,0.0057,0.7190,181.9025,False
2,AreaCode,0.0006,No Drift,0.0109,0.0610,27.5768,False
7,TenureinMonths,0.0006,No Drift,0.0078,0.3202,0.2538,False
10,MonthlyCharge,0.0006,No Drift,0.0070,0.4692,0.2897,False
3,Latitude,0.0005,No Drift,0.0099,0.1069,0.0420,False


In [47]:
drift_results[drift_results["Drift_Detected"] == True]

,Feature,PSI,PSI_Severity,KS_Statistic,p_value,Wasserstein_Distance,Drift_Detected
0,UserAge,2.5736,Significant Drift,0.3639,0.0,14.8781,True


## Observations

No drift exists between two samples drawn from the same dataset.